# What happened to campus speech trends in 2026?

This notebook extends the five figures in [“Violence is up and tolerance is down. What comes next?”](https://expression.fire.org/p/violence-is-up-and-tolerance-is-down) through the 2026 survey wave (the data used for the 2027 College Free Speech Rankings).

## Methods in brief

- The source is `cfsrALL_2027.csv`; this package contains a reduced, de-identified extract with only the fields used below.
- Violence, shoutdown, comfort, and difficult-topic results use the national survey weight (`weight`). In the 2026 rows, `weight` and `fouryearweight` are identical.
- “At least rarely acceptable” combines response codes 1–3; code 4 is “Never acceptable.”
- Comfort is the survey-weighted mean on its original 1–4 scale, from “Very uncomfortable” to “Very comfortable.”
- Difficult topics are weighted shares selecting each option (1 versus 0).
- To preserve exact continuity with the earlier chart, speaker tolerance uses the same unweighted complete-case method, gender and party-ID filters, year-specific speaker batteries, and 0–100 scaling as the source notebook. A weighted sensitivity estimate is exported as well.
- Confidence intervals are simple Kish-effective-sample-size approximations. They do not capture all uncertainty from the opt-in survey design, weighting, repeated respondents, or questionnaire effects.

All invalid, missing, and “not in survey” codes are excluded measure by measure. Output tables retain raw valid counts and effective sample sizes where applicable.

In [1]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd
from IPython.display import display

BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / "data" / "trends_followup_analysis_data_2020_2026.csv.zip"
OUT_DIR = BASE_DIR / "trends_article_outputs"
OUT_DIR.mkdir(exist_ok=True)

START_YEAR = 2020
END_YEAR = 2026
YEAR_COL = "datayear"
WEIGHT_COL = "weight"

FIRE = {
    "navy": "#003d6e",
    "blue": "#00a6de",
    "light_blue": "#85c7e3",
    "dark": "#31261d",
    "red": "#cb333b",
    "orange": "#f24f00",
    "gold": "#f7a30a",
    "green": "#009a44",
    "purple": "#6a1b9a",
    "gray": "#4b5563",
    "grid": "#d1d5db",
}

CHLOE_QUESTION_SETS = {
    2021: {
        "right": ["spk_abortion", "spk_trans", "spk_blm", "spk_lockdown"],
        "left": ["spk_whites", "spk_loot", "spk_abolishpolice", "spk_religlib"],
    },
    2022: {
        "right": ["spk_abortion", "spk_trans", "spk_blm", "spk_election"],
        "left": ["spk_whites", "spk_guns", "spk_religlib", "spk_immigration"],
    },
    2023: {
        "right": ["spk_abortion", "spk_trans", "spk_blm"],
        "left": ["spk_whites", "spk_guns", "spk_religlib"],
    },
    2024: {
        "right": ["spk_abortion", "spk_trans", "spk_blm"],
        "left": ["spk_cathped", "spk_policekkk", "spk_kidtrans"],
    },
    2025: {
        "right": ["spk_abortion", "spk_trans", "spk_blm"],
        "left": ["spk_cathped", "spk_policekkk", "spk_kidtrans"],
    },
    2026: {
        "right": ["spk_abortion", "spk_trans", "spk_blm"],
        "left": ["spk_cathped", "spk_policekkk", "spk_kidtrans"],
    },
}

COMFORT_ITEMS = {
    "cf_pubprof": "Publicly disagree with a professor",
    "cf_wrtprof": "Disagree in a written assignment",
    "cf_inclass": "Express views in class",
    "cf_quad": "Discuss views in a campus common space",
    "cf_socmedia": "Post an unpopular opinion on social media",
}

DIFFICULT_TOPIC_ITEMS = {
    "tk_ipc": "Israeli-Palestinian conflict",
    "tk_abortion": "Abortion",
    "tk_preselec": "2024 presidential election",
    "tk_trans": "Transgender rights",
}

NEW_2026_TOPIC_ITEMS = {
    "tk_trump": "Donald Trump (new in 2026)",
    "tk_charliekirk": "Charlie Kirk (new in 2026)",
}

data = pd.read_csv(DATA_PATH, low_memory=False)
source_metadata = json.loads((BASE_DIR / "data" / "source_metadata.json").read_text())
for column in data.columns:
    data[column] = pd.to_numeric(data[column], errors="coerce")
data = data[data[YEAR_COL].between(START_YEAR, END_YEAR)].copy()

assert len(data) == 347_369
assert data.groupby(YEAR_COL).size().to_dict() == {
    2020: 20_002,
    2021: 37_104,
    2022: 44_847,
    2023: 55_102,
    2024: 58_807,
    2025: 68_510,
    2026: 62_997,
}
rows_2026 = data[data[YEAR_COL].eq(2026)]
assert np.allclose(rows_2026["weight"], rows_2026["fouryearweight"])
dates_2026 = pd.to_datetime([
    source_metadata["survey_start_2026"], source_metadata["survey_end_2026"]
])
print(
    f"Loaded {len(data):,} rows for {START_YEAR}–{END_YEAR}; "
    f"2026 field dates are {dates_2026.min():%b. %d}–{dates_2026.max():%b. %d, %Y}."
)

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 18,
    "axes.titleweight": "bold",
    "axes.labelsize": 12,
    "axes.labelweight": "bold",
    "legend.fontsize": 10.0,
})

Loaded 347,369 rows for 2020–2026; 2026 field dates are Jan. 02–Jun. 03, 2026.


In [2]:
def weighted_stats(values, weights):
    """Weighted mean plus a simple Kish-n standard-error approximation."""
    values = pd.to_numeric(values, errors="coerce")
    weights = pd.to_numeric(weights, errors="coerce")
    valid = values.notna() & weights.notna() & weights.gt(0)
    x = values[valid].astype(float).to_numpy()
    w = weights[valid].astype(float).to_numpy()
    if len(x) == 0:
        return {
            "estimate": np.nan, "n_valid": 0, "sum_weight": np.nan,
            "kish_n": np.nan, "se": np.nan, "ci_low": np.nan, "ci_high": np.nan,
        }
    estimate = np.average(x, weights=w)
    kish_n = w.sum() ** 2 / np.square(w).sum()
    variance = np.average(np.square(x - estimate), weights=w)
    se = np.sqrt(variance / kish_n)
    return {
        "estimate": estimate,
        "n_valid": len(x),
        "sum_weight": w.sum(),
        "kish_n": kish_n,
        "se": se,
        "ci_low": estimate - 1.96 * se,
        "ci_high": estimate + 1.96 * se,
    }


def weighted_percent_stats(condition, weights):
    stats = weighted_stats(pd.Series(condition, index=weights.index, dtype=float), weights)
    for key in ["estimate", "se", "ci_low", "ci_high"]:
        stats[key] *= 100
    return stats


def style_axes(ax, years, *, ylim, ylabel, percent=False):
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)
    ax.set_xticks(list(years))
    ax.set_ylim(*ylim)
    ax.set_axisbelow(True)
    ax.grid(axis="y", color=FIRE["grid"], linewidth=0.9, alpha=0.8)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(axis="both", length=0, pad=7)
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight("semibold")
    if percent:
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))


def label_last_two(ax, frame, value_col, *, color, suffix="%", decimals=1):
    for row in frame.tail(2).itertuples(index=False):
        ax.annotate(
            f"{getattr(row, value_col):.{decimals}f}{suffix}",
            (row.year, getattr(row, value_col)),
            xytext=(0, 9), textcoords="offset points",
            ha="center", va="bottom", fontsize=9.5,
            fontweight="semibold", color=color,
        )


def finish_plot(fig, ax, *, title, filename, source_years, legend=None, note=None):
    ax.set_title(title, pad=18)
    if legend:
        ax.legend(**legend)
    credit = (
        "Chart: Chapin Lenthall-Cleary\n"
        f"Source: FIRE's College Free Speech Rankings Survey ({source_years} data)"
    )
    if note:
        credit = f"{note}\n{credit}"
    fig.text(0.98, 0.018, credit, ha="right", va="bottom", fontsize=8.8, color=FIRE["gray"])
    fig.tight_layout(rect=[0.02, 0.08, 0.98, 0.97])
    path = OUT_DIR / filename
    fig.savefig(path, dpi=240, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Saved: {path}")

In [3]:
# 1. Stated acceptance of violence against speakers
violence_rows = []
for year, year_df in data.groupby(YEAR_COL, sort=True):
    valid = year_df["act_viol"].isin([1, 2, 3, 4])
    sub = year_df.loc[valid]
    stats = weighted_percent_stats(sub["act_viol"].isin([1, 2, 3]), sub[WEIGHT_COL])
    violence_rows.append({"year": int(year), **stats})
violence_trend = pd.DataFrame(violence_rows)
violence_trend.to_csv(OUT_DIR / "violence_acceptability_trend_2020_2026.csv", index=False)
display(violence_trend.round(2))

assert np.isclose(violence_trend.loc[violence_trend.year.eq(2025), "estimate"].item(), 33.6223435523)
assert np.isclose(violence_trend.loc[violence_trend.year.eq(2026), "estimate"].item(), 30.4861675708)

fig, ax = plt.subplots(figsize=(11.5, 6.6))
ax.plot(
    violence_trend["year"], violence_trend["estimate"], color=FIRE["dark"],
    linewidth=3.2, marker="o", markersize=8, markeredgecolor="white", markeredgewidth=1.4,
)
style_axes(ax, violence_trend["year"], ylim=(0, 40), ylabel="Violence is at least rarely acceptable", percent=True)
ax.set_yticks([0, 10, 20, 30, 40])
label_last_two(ax, violence_trend, "estimate", color=FIRE["dark"])
finish_plot(
    fig, ax, title="Stated acceptance of violence fell in 2026",
    filename="violence_acceptability_trend_2020_2026.png", source_years="2020–2026",
)

,year,estimate,n_valid,sum_weight,kish_n,se,ci_low,ci_high
0,2020,17.95,19899,19963.48,13048.38,0.34,17.29,18.60
1,2021,23.81,37104,37060.00,24915.18,0.27,23.28,24.34
2,2022,20.36,44847,44788.41,21813.48,0.27,19.82,20.89
3,2023,26.91,55102,55058.09,31961.71,0.25,26.42,27.39
4,2024,31.79,58807,58807.00,35223.14,0.25,31.30,32.27
5,2025,33.62,68510,68455.02,36483.18,0.25,33.14,34.11
6,2026,30.49,62996,62996.18,43293.65,0.22,30.05,30.92


Saved: /Users/chapin.lenthall-cleary/Documents/codex_datapost_test_6/trends_2026_followup/trends_article_outputs/violence_acceptability_trend_2020_2026.png


/var/folders/6p/86hvnths0bb7nsydcyhcx87m0000gp/T/ipykernel_55331/997518904.py:76: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
# 2. Stated acceptance of shouting down speakers
shoutdown_rows = []
for year, year_df in data.groupby(YEAR_COL, sort=True):
    valid = year_df["act_shout"].isin([1, 2, 3, 4])
    sub = year_df.loc[valid]
    stats = weighted_percent_stats(sub["act_shout"].isin([1, 2, 3]), sub[WEIGHT_COL])
    shoutdown_rows.append({"year": int(year), **stats})
shoutdown_trend = pd.DataFrame(shoutdown_rows)
shoutdown_trend.to_csv(OUT_DIR / "shoutdown_trend_2020_2026.csv", index=False)
display(shoutdown_trend.round(2))

assert np.isclose(shoutdown_trend.loc[shoutdown_trend.year.eq(2025), "estimate"].item(), 71.7105287163)
assert np.isclose(shoutdown_trend.loc[shoutdown_trend.year.eq(2026), "estimate"].item(), 68.9139659828)

fig, ax = plt.subplots(figsize=(11.5, 6.6))
ax.plot(
    shoutdown_trend["year"], shoutdown_trend["estimate"], color=FIRE["navy"],
    linewidth=3.2, marker="o", markersize=8, markeredgecolor="white", markeredgewidth=1.4,
)
style_axes(ax, shoutdown_trend["year"], ylim=(0, 80), ylabel="Shoutdowns are at least rarely acceptable", percent=True)
ax.set_yticks([0, 20, 40, 60, 80])
label_last_two(ax, shoutdown_trend, "estimate", color=FIRE["navy"])
finish_plot(
    fig, ax, title="Stated acceptance of shoutdowns fell too",
    filename="shoutdown_trend_2020_2026.png", source_years="2020–2026",
)

,year,estimate,n_valid,sum_weight,kish_n,se,ci_low,ci_high
0,2020,61.02,19889,19959.37,13042.24,0.43,60.18,61.86
1,2021,66.03,37104,37060.00,24915.18,0.30,65.44,66.62
2,2022,62.31,44846,44787.11,21812.61,0.33,61.67,62.95
3,2023,63.40,55101,55057.68,31961.29,0.27,62.87,63.93
4,2024,68.50,58807,58807.00,35223.14,0.25,68.01,68.98
5,2025,71.71,68510,68455.02,36483.18,0.24,71.25,72.17
6,2026,68.91,62997,62996.59,43294.13,0.22,68.48,69.35


Saved: /Users/chapin.lenthall-cleary/Documents/codex_datapost_test_6/trends_2026_followup/trends_article_outputs/shoutdown_trend_2020_2026.png


/var/folders/6p/86hvnths0bb7nsydcyhcx87m0000gp/T/ipykernel_55331/997518904.py:76: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# 3. Left- and right-wing speaker tolerance
def scale_items_to_pct(sum_values, k):
    return (sum_values - k) / (3 * k) * 100.0


def build_partyid_like_source(frame):
    fire = pd.to_numeric(frame["fire_partyid"], errors="coerce")
    cp = pd.to_numeric(frame["cp_partyid"], errors="coerce")
    partyid = np.where(fire.between(1, 7), fire, np.where(cp.between(1, 7), cp, np.nan))
    partyid = np.where(pd.isna(partyid) & ((fire == 8) | (cp == 8)), 8, partyid)
    return pd.Series(partyid, index=frame.index).astype("Int64")


speaker_data = data.copy()
speaker_data["partyid"] = build_partyid_like_source(speaker_data)
tolerance_rows = []
tolerance_sensitivity_rows = []
for year in range(2021, END_YEAR + 1):
    right_items = CHLOE_QUESTION_SETS[year]["right"]
    left_items = CHLOE_QUESTION_SETS[year]["left"]
    speaker_items = right_items + left_items
    year_df = speaker_data[
        speaker_data[YEAR_COL].eq(year)
        & speaker_data["gender_bin"].isin([1, 2])
        & speaker_data["partyid"].isin(range(1, 8))
    ].copy()
    in_range = year_df[speaker_items].ge(1) & year_df[speaker_items].le(4)
    year_df = year_df[in_range.all(axis=1)].copy()
    year_df["right_pct"] = scale_items_to_pct(year_df[right_items].sum(axis=1), len(right_items))
    year_df["left_pct"] = scale_items_to_pct(year_df[left_items].sum(axis=1), len(left_items))
    row = {"year": year, "n_complete": len(year_df)}
    sensitivity = {"year": year, "n_complete": len(year_df)}
    for side in ["left", "right"]:
        values = year_df[f"{side}_pct"]
        estimate = values.mean()
        se = values.sem()
        row.update({
            f"avg_{side}_tolerance": estimate,
            f"se_{side}": se,
            f"ci_low_{side}": estimate - 1.96 * se,
            f"ci_high_{side}": estimate + 1.96 * se,
        })
        sensitivity[f"weighted_{side}_tolerance"] = weighted_stats(values, year_df[WEIGHT_COL])["estimate"]
    tolerance_rows.append(row)
    tolerance_sensitivity_rows.append(sensitivity)

tolerance_trends = pd.DataFrame(tolerance_rows)
tolerance_sensitivity = pd.DataFrame(tolerance_sensitivity_rows)
tolerance_trends.to_csv(OUT_DIR / "speaker_tolerance_trends_source_method_2021_2026.csv", index=False)
tolerance_sensitivity.to_csv(OUT_DIR / "speaker_tolerance_weighted_sensitivity_2021_2026.csv", index=False)
display(tolerance_trends.round(2))
display(tolerance_sensitivity.round(2))

assert np.isclose(tolerance_trends.loc[tolerance_trends.year.eq(2025), "avg_left_tolerance"].item(), 41.2149064171)
assert np.isclose(tolerance_trends.loc[tolerance_trends.year.eq(2026), "avg_right_tolerance"].item(), 32.5423523173)

fig, ax = plt.subplots(figsize=(11.5, 6.8))
ax.plot(
    tolerance_trends["year"], tolerance_trends["avg_left_tolerance"], marker="o",
    linewidth=3.0, markersize=8, markeredgecolor="white", markeredgewidth=1.3,
    color=FIRE["blue"], label="Tolerance for left-wing speakers",
)
ax.plot(
    tolerance_trends["year"], tolerance_trends["avg_right_tolerance"], marker="o",
    linewidth=3.0, markersize=8, markeredgecolor="white", markeredgewidth=1.3,
    color=FIRE["red"], label="Tolerance for right-wing speakers",
)
style_axes(ax, tolerance_trends["year"], ylim=(0, 70), ylabel="Average tolerance score", percent=True)
ax.set_yticks([0, 10, 20, 30, 40, 50, 60, 70])
for side, color in [("left", FIRE["blue"]), ("right", FIRE["red"])]:
    label_last_two(ax, tolerance_trends.rename(columns={f"avg_{side}_tolerance": "value"}), "value", color=color)
finish_plot(
    fig, ax, title="Speaker tolerance did not recover in 2026",
    filename="speaker_tolerance_trends_2021_2026.png", source_years="2021–2026",
    legend={"frameon": False, "loc": "upper right"},
    note="Left-wing speaker battery changed between 2023 and 2024.",
)

,year,n_complete,avg_left_tolerance,se_left,ci_low_left,ci_high_left,avg_right_tolerance,se_right,ci_low_right,ci_high_right
0,2021,33298,51.87,0.16,51.56,52.18,29.25,0.16,28.93,29.57
1,2022,38119,59.78,0.13,59.53,60.03,32.42,0.15,32.12,32.72
2,2023,46886,58.55,0.12,58.31,58.78,34.53,0.14,34.25,34.80
3,2024,48781,47.33,0.12,47.10,47.56,37.32,0.13,37.06,37.58
4,2025,59840,41.21,0.10,41.01,41.42,33.13,0.11,32.91,33.35
5,2026,56779,41.19,0.11,40.98,41.40,32.54,0.12,32.32,32.77


,year,n_complete,weighted_left_tolerance,weighted_right_tolerance
0,2021,33298,51.62,32.02
1,2022,38119,59.89,34.13
2,2023,46886,58.78,36.21
3,2024,48781,48.22,39.74
4,2025,59840,41.67,34.41
5,2026,56779,41.17,33.53


Saved: /Users/chapin.lenthall-cleary/Documents/codex_datapost_test_6/trends_2026_followup/trends_article_outputs/speaker_tolerance_trends_2021_2026.png


/var/folders/6p/86hvnths0bb7nsydcyhcx87m0000gp/T/ipykernel_55331/997518904.py:76: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# 4. Comfort expressing controversial or unpopular opinions
comfort_rows = []
for year, year_df in data.groupby(YEAR_COL, sort=True):
    for item, label in COMFORT_ITEMS.items():
        valid = year_df[item].isin([1, 2, 3, 4])
        if not valid.any():
            continue
        stats = weighted_stats(year_df.loc[valid, item], year_df.loc[valid, WEIGHT_COL])
        comfort_rows.append({"year": int(year), "question": label, **stats})
comfort_trends = pd.DataFrame(comfort_rows)
comfort_trends.to_csv(OUT_DIR / "comfort_trends_2020_2026.csv", index=False)
display(comfort_trends.pivot(index="year", columns="question", values="estimate").round(2))

comfort_changes = (
    comfort_trends[comfort_trends.year.isin([2025, 2026])]
    .pivot(index="question", columns="year", values="estimate")
)
comfort_changes["change"] = comfort_changes[2026] - comfort_changes[2025]
assert comfort_changes["change"].gt(0).all()

comfort_colors = {
    "Publicly disagree with a professor": FIRE["navy"],
    "Disagree in a written assignment": FIRE["orange"],
    "Express views in class": FIRE["green"],
    "Discuss views in a campus common space": FIRE["red"],
    "Post an unpopular opinion on social media": FIRE["purple"],
}
fig, ax = plt.subplots(figsize=(12, 7.2))
for question in COMFORT_ITEMS.values():
    plot_df = comfort_trends[comfort_trends["question"].eq(question)]
    ax.plot(
        plot_df["year"], plot_df["estimate"], marker="o", markersize=7.5,
        markeredgecolor="white", markeredgewidth=1.2, linewidth=2.8,
        color=comfort_colors[question], label=question,
    )
style_axes(ax, range(START_YEAR, END_YEAR + 1), ylim=(1, 4), ylabel="Average comfort score")
ax.set_yticks([1, 2, 3, 4])
ax.set_yticklabels([
    "Very uncomfortable (1)", "Somewhat uncomfortable (2)",
    "Somewhat comfortable (3)", "Very comfortable (4)",
])
finish_plot(
    fig, ax, title="Comfort improved in every setting in 2026",
    filename="comfort_trends_2020_2026.png", source_years="2020–2026",
    legend={"frameon": False, "loc": "upper center", "bbox_to_anchor": (0.5, 1.0), "ncol": 2},
)

question,Disagree in a written assignment,Discuss views in a campus common space,Express views in class,Post an unpopular opinion on social media,Publicly disagree with a professor
year,,,,,
2020,NaN,NaN,2.84,2.20,2.38
2021,2.65,2.68,2.50,2.25,2.27
2022,2.66,2.70,2.50,2.26,2.28
2023,2.39,2.42,2.30,2.02,2.08
2024,2.47,2.47,2.40,2.10,2.25
2025,2.48,2.52,2.44,2.12,2.27
2026,2.55,2.62,2.52,2.20,2.35


Saved: /Users/chapin.lenthall-cleary/Documents/codex_datapost_test_6/trends_2026_followup/trends_article_outputs/comfort_trends_2020_2026.png


/var/folders/6p/86hvnths0bb7nsydcyhcx87m0000gp/T/ipykernel_55331/997518904.py:76: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# 5. Difficult topics
topic_rows = []
for year, year_df in data.groupby(YEAR_COL, sort=True):
    for item, label in {**DIFFICULT_TOPIC_ITEMS, **NEW_2026_TOPIC_ITEMS}.items():
        valid = year_df[item].isin([0, 1])
        if not valid.any():
            continue
        stats = weighted_percent_stats(year_df.loc[valid, item].eq(1), year_df.loc[valid, WEIGHT_COL])
        topic_rows.append({"year": int(year), "topic": label, "variable": item, **stats})
topic_trends = pd.DataFrame(topic_rows)
topic_trends.to_csv(OUT_DIR / "difficult_topic_trends_2020_2026.csv", index=False)

topic_2026 = topic_trends[topic_trends.year.eq(2026)].sort_values("estimate", ascending=False)
display(topic_2026.round(2))
display(
    topic_trends[topic_trends.variable.isin(DIFFICULT_TOPIC_ITEMS)]
    .pivot(index="year", columns="topic", values="estimate")
    .round(1)
)

tracked_changes = (
    topic_trends[topic_trends.year.isin([2025, 2026]) & topic_trends.variable.isin(DIFFICULT_TOPIC_ITEMS)]
    .pivot(index="topic", columns="year", values="estimate")
)
tracked_changes["change"] = tracked_changes[2026] - tracked_changes[2025]
assert tracked_changes["change"].lt(0).all()

topic_colors = {
    "Israeli-Palestinian conflict": FIRE["red"],
    "Abortion": FIRE["orange"],
    "2024 presidential election": FIRE["navy"],
    "Transgender rights": FIRE["purple"],
}
fig, ax = plt.subplots(figsize=(12, 7.2))
for topic in DIFFICULT_TOPIC_ITEMS.values():
    plot_df = topic_trends[topic_trends["topic"].eq(topic)].sort_values("year")
    ax.plot(
        plot_df["year"], plot_df["estimate"], label=topic, color=topic_colors[topic],
        linewidth=3.0, marker="o", markersize=8, markeredgecolor="white", markeredgewidth=1.4,
    )
for variable, color, marker in [("tk_trump", FIRE["gold"], "D"), ("tk_charliekirk", FIRE["green"], "s")]:
    point = topic_trends[(topic_trends.year.eq(2026)) & (topic_trends.variable.eq(variable))].iloc[0]
    ax.scatter(point.year, point.estimate, s=85, marker=marker, color=color, edgecolor="white", linewidth=1.2, zorder=5, label=point.topic)

style_axes(ax, range(2020, 2027), ylim=(0, 60), ylabel="Share identifying the topic as difficult to discuss", percent=True)
ax.set_yticks([0, 10, 20, 30, 40, 50, 60])
finish_plot(
    fig, ax, title="The Israeli-Palestinian conflict remains the hardest topic",
    filename="difficult_topics_trend_2020_2026.png", source_years="2020–2026",
    legend={"frameon": False, "loc": "upper left", "ncol": 2},
    note="Donald Trump and Charlie Kirk were first offered as response options in 2026.",
)

,year,topic,variable,estimate,n_valid,sum_weight,kish_n,se,ci_low,ci_high
20,2026,Israeli-Palestinian conflict,tk_ipc,49.60,62997,62996.59,43294.13,0.24,49.13,50.07
24,2026,Donald Trump (new in 2026),tk_trump,45.60,62997,62996.59,43294.13,0.24,45.13,46.06
25,2026,Charlie Kirk (new in 2026),tk_charliekirk,43.89,62997,62996.59,43294.13,0.24,43.42,44.36
21,2026,Abortion,tk_abortion,43.49,62997,62996.59,43294.13,0.24,43.02,43.95
23,2026,Transgender rights,tk_trans,37.81,62997,62996.59,43294.13,0.23,37.36,38.27
22,2026,2024 presidential election,tk_preselec,28.79,62997,62996.59,43294.13,0.22,28.36,29.21


topic,2024 presidential election,Abortion,Israeli-Palestinian conflict,Transgender rights
year,,,,
2020,NaN,45.1,29.7,39.5
2021,NaN,45.7,30.2,41.9
2022,NaN,48.9,30.9,44.2
2023,NaN,49.4,25.8,41.9
2024,30.9,45.0,54.9,41.6
2025,42.2,45.6,53.3,40.8
2026,28.8,43.5,49.6,37.8


Saved: /Users/chapin.lenthall-cleary/Documents/codex_datapost_test_6/trends_2026_followup/trends_article_outputs/difficult_topics_trend_2020_2026.png


/var/folders/6p/86hvnths0bb7nsydcyhcx87m0000gp/T/ipykernel_55331/997518904.py:76: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# 6. Year-over-year audit and subgroup check
def value(frame, year, column="estimate"):
    return frame.loc[frame.year.eq(year), column].item()


yoy_rows = [
    {
        "measure": "Violence at least rarely acceptable",
        "units": "percentage points",
        "estimate_2025": value(violence_trend, 2025),
        "estimate_2026": value(violence_trend, 2026),
    },
    {
        "measure": "Shoutdowns at least rarely acceptable",
        "units": "percentage points",
        "estimate_2025": value(shoutdown_trend, 2025),
        "estimate_2026": value(shoutdown_trend, 2026),
    },
    {
        "measure": "Tolerance for left-wing speakers",
        "units": "points on 0–100 scale",
        "estimate_2025": value(tolerance_trends, 2025, "avg_left_tolerance"),
        "estimate_2026": value(tolerance_trends, 2026, "avg_left_tolerance"),
    },
    {
        "measure": "Tolerance for right-wing speakers",
        "units": "points on 0–100 scale",
        "estimate_2025": value(tolerance_trends, 2025, "avg_right_tolerance"),
        "estimate_2026": value(tolerance_trends, 2026, "avg_right_tolerance"),
    },
]
for item, label in COMFORT_ITEMS.items():
    frame = comfort_trends[comfort_trends.question.eq(label)]
    yoy_rows.append({
        "measure": f"Comfort: {label}",
        "units": "points on 1–4 scale",
        "estimate_2025": value(frame, 2025),
        "estimate_2026": value(frame, 2026),
    })
for item, label in DIFFICULT_TOPIC_ITEMS.items():
    frame = topic_trends[topic_trends.variable.eq(item)]
    yoy_rows.append({
        "measure": f"Difficult topic: {label}",
        "units": "percentage points",
        "estimate_2025": value(frame, 2025),
        "estimate_2026": value(frame, 2026),
    })
yoy = pd.DataFrame(yoy_rows)
yoy["change_2025_to_2026"] = yoy["estimate_2026"] - yoy["estimate_2025"]
yoy.to_csv(OUT_DIR / "year_over_year_2025_2026.csv", index=False)
display(yoy.round(2))

party_labels = {
    1: "Strong Democrat", 2: "Democrat", 3: "Lean Democrat", 4: "Independent",
    5: "Lean Republican", 6: "Republican", 7: "Strong Republican",
}
party_rows = []
for year in [2025, 2026]:
    year_df = speaker_data[speaker_data.year.eq(year)] if "year" in speaker_data.columns else speaker_data[speaker_data.datayear.eq(year)]
    for party_code, party_label in party_labels.items():
        group = year_df[year_df.partyid.eq(party_code)]
        for item, label in [("act_viol", "Violence"), ("act_shout", "Shoutdown")]:
            valid = group[item].isin([1, 2, 3, 4])
            stats = weighted_percent_stats(group.loc[valid, item].isin([1, 2, 3]), group.loc[valid, WEIGHT_COL])
            party_rows.append({
                "year": year, "party_code": party_code, "party": party_label,
                "measure": label, **stats,
            })
party_trends = pd.DataFrame(party_rows)
party_trends.to_csv(OUT_DIR / "partyid_acceptability_2025_2026.csv", index=False)
party_change = party_trends.pivot(index=["party_code", "party", "measure"], columns="year", values="estimate")
party_change["change"] = party_change[2026] - party_change[2025]
display(party_change.round(1))
assert party_change["change"].lt(0).all()

summary = {
    "source_rows": int(len(data)),
    "rows_2026": int(data[YEAR_COL].eq(2026).sum()),
    "violence_2025": value(violence_trend, 2025),
    "violence_2026": value(violence_trend, 2026),
    "shoutdown_2025": value(shoutdown_trend, 2025),
    "shoutdown_2026": value(shoutdown_trend, 2026),
    "left_tolerance_2025": value(tolerance_trends, 2025, "avg_left_tolerance"),
    "left_tolerance_2026": value(tolerance_trends, 2026, "avg_left_tolerance"),
    "right_tolerance_2025": value(tolerance_trends, 2025, "avg_right_tolerance"),
    "right_tolerance_2026": value(tolerance_trends, 2026, "avg_right_tolerance"),
    "speaker_complete_cases_2026": int(value(tolerance_trends, 2026, "n_complete")),
    "ipc_2025": value(topic_trends[topic_trends.variable.eq("tk_ipc")], 2025),
    "ipc_2026": value(topic_trends[topic_trends.variable.eq("tk_ipc")], 2026),
    "trump_2026": value(topic_trends[topic_trends.variable.eq("tk_trump")], 2026),
    "charlie_kirk_2026": value(topic_trends[topic_trends.variable.eq("tk_charliekirk")], 2026),
    "survey_start_2026": str(dates_2026.min()),
    "survey_end_2026": str(dates_2026.max()),
}
(OUT_DIR / "results_summary.json").write_text(json.dumps(summary, indent=2) + "\n")
print("All validation checks passed.")

,measure,units,estimate_2025,estimate_2026,change_2025_to_2026
0,Violence at least rarely acceptable,percentage points,33.62,30.49,-3.14
1,Shoutdowns at least rarely acceptable,percentage points,71.71,68.91,-2.80
2,Tolerance for left-wing speakers,points on 0–100 scale,41.21,41.19,-0.03
3,Tolerance for right-wing speakers,points on 0–100 scale,33.13,32.54,-0.58
4,Comfort: Publicly disagree with a professor,points on 1–4 scale,2.27,2.35,0.08
5,Comfort: Disagree in a written assignment,points on 1–4 scale,2.48,2.55,0.07
6,Comfort: Express views in class,points on 1–4 scale,2.44,2.52,0.08
7,Comfort: Discuss views in a campus common space,points on 1–4 scale,2.52,2.62,0.11
8,Comfort: Post an unpopular opinion on social m...,points on 1–4 scale,2.12,2.20,0.08
9,Difficult topic: Israeli-Palestinian conflict,percentage points,53.25,49.60,-3.66


year                                    2025  2026  change
party_code party             measure                      
1          Strong Democrat   Shoutdown  77.9  76.4    -1.6
                             Violence   31.2  29.8    -1.4
2          Democrat          Shoutdown  76.1  73.3    -2.8
                             Violence   31.5  28.6    -3.0
3          Lean Democrat     Shoutdown  76.2  72.2    -4.0
                             Violence   30.7  27.1    -3.7
4          Independent       Shoutdown  69.6  65.0    -4.6
                             Violence   38.1  34.9    -3.2
5          Lean Republican   Shoutdown  61.2  57.4    -3.8
                             Violence   29.2  25.3    -3.9
6          Republican        Shoutdown  62.0  58.6    -3.5
                             Violence   32.0  30.2    -1.8
7          Strong Republican Shoutdown  57.3  53.5    -3.8
                             Violence   35.4  32.3    -3.0

All validation checks passed.
